In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ClaudeParams:
    model:                str
    max_tokens:           int
    confidence_threshold: float

@dataclass(frozen=True)
class ClaudeValidationConfig:
    params: ClaudeParams

In [3]:
from core.constants import *
from core.utils import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_claude_config(self) -> ClaudeValidationConfig:
        params = self.params.claude_params
        
        return ClaudeValidationConfig(
            params = ClaudeParams(
                model       = params.model,
                max_tokens  = params.max_tokens,
                confidence_threshold = params.confidence_threshold
            )
        )

In [4]:
import os, sys, anthropic
from pydantic import BaseModel, Field
from enum import Enum
from core.logging import logger
from core.exception import CustomException

from dotenv import load_dotenv
load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

class ActionType(str, Enum):
    do_nothing           = "do_nothing"
    trigger_retraining   = "trigger_retraining"
    switch_to_lighter    = "switch_to_lighter_model"
    restart_service      = "restart_service"
    scale_out            = "scale_out_service"
    scale_in             = "scale_in_service"
    swap_model           = "swap_model_version"
    
class ExecuteAction(BaseModel):
    action:     ActionType  = Field(description="Action to execute")
    reasoning:  str         = Field(description="Why this action was chosen")

class ClaudeValidation:
    def __init__(self, config: ClaudeValidationConfig):
        self.config  = config
        self.client  = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.tools   = self._get_tools()
        
    def _get_tools(self):
        return [anthropic.types.ToolParam(
            name="execute_action",
            description="Execute a system action when metrics indicate a problem",
            input_schema=ExecuteAction.model_json_schema()
        )]
    
    def claude_validate(self, metrics: dict, dqn_suggestion: str, dqn_confidence: float) -> dict:
        try:
            response  = client.messages.create(
                model = self.config.params.model,
                max_tokens = self.config.params.max_tokens,
                tools = self.tools,
                messages = [{
                    "role": "user",
                    "content": self._build_prompt(metrics, dqn_suggestion, dqn_confidence)
                }]
            )       
        
            for block in response.content:
                if block.type == "tool_use":
                    return {
                        "action":    block.input["action"],
                        "reasoning": block.input["reasoning"]
                    }

            return {"action": dqn_suggestion, "reasoning": "Claude fallback to DQN"}
        except Exception as e:
            raise CustomException(e, sys)
        
    def _build_prompt(self, metrics: dict, dqn_suggestion: str, dqn_confidence: float) -> str:
        return f"""
        You are an AI system monitor for a medical image segmentation service.
        Given these metrics, decide the best action.

        Current metrics:
        - CPU:         {metrics['cpu']}%
        - RAM:         {metrics['ram']}%
        - Latency:     {metrics['latency']}s
        - Drift score: {metrics['drift_score']} (threshold: {self.config.params.confidence_threshold})

        DQN suggestion: {dqn_suggestion} (confidence: {dqn_confidence:.2f})
        Low confidence means DQN is uncertain — use your judgment.

        Choose the most appropriate action and explain why.
        """.strip()

In [5]:
try:
    config = ConfigurationManger()
    claude_config = config.get_claude_config()
    claude_validate = ClaudeValidation(config=claude_config)
    results = claude_validate.claude_validate(
        metrics = {
            "cpu":         95.0,
            "ram":         40.0,
            "latency":     0.1,
            "drift_score": 0.05
        },
        dqn_suggestion="trigger_retraining",
        dqn_confidence=0.9 
    )
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-28 20:52:27,205: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-28 20:52:27,216: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-28 20:52:27,218: INFO: __init__: created directory at: artifacts]
[2026-04-28 20:52:32,792: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
scale_out_service
CPU utilization is critically high at 95%, indicating the service is under severe computational stress and at risk of degradation or failure. However, latency remains excellent (0.1s) and drift score is very low (0.05), indicating the model quality is not the issue. Scaling out will distribute the workload across additional instances, reducing CPU pressure on individual nodes and maintaining service availability and performance. This is more appropriate than retraining, which the DQN suggested but isn't justified by the low drift score.
